# 01 · Generation and KV caching

Companion notebook for [Chapter 2](../chapters/ch02-generation.md).

**You will:** tokenise text, generate greedily, watch the naive loop's cost grow,
add a KV cache, and measure the speed-up on your own hardware.

## 0 · Setup

Run this cell first. It picks a device and imports everything the notebook needs.

If you get `ModuleNotFoundError: reasoning_from_scratch`, your kernel is not the
course environment — see the [setup page](../start/setup.md).

In [ ]:
import torch
from reasoning_from_scratch.ch02 import get_device

device = get_device()
print("torch", torch.__version__)

## 1 · Tokens

The model has never seen a "word". It sees integers into a 151,936-entry
vocabulary.

In [ ]:
from pathlib import Path
from reasoning_from_scratch.qwen3 import Qwen3Tokenizer, download_qwen3_small

download_qwen3_small(kind="base", tokenizer_only=True, out_dir="qwen3")
tokenizer = Qwen3Tokenizer(
    tokenizer_file_path=Path("qwen3") / "tokenizer-base.json"
)

ids = tokenizer.encode("Explain large language models.")
print("num tokens:", len(ids))
for i in ids:
    print(f"{i:>7} --> {tokenizer.decode([i])!r}")

print()
print("round trip:", repr(tokenizer.decode(ids)))

**Look at the split.** Words are not tokens. `models` may be one token or several;
punctuation is usually its own token. Everything downstream — context limits, KV
cache size, `max_new_tokens` — is measured in these units, not words.

## 2 · Load the model

In [ ]:
from pathlib import Path
from reasoning_from_scratch.qwen3 import (
    Qwen3Model, QWEN_CONFIG_06_B, Qwen3Tokenizer, download_qwen3_small,
)

download_qwen3_small(kind="base", out_dir="qwen3")     # 1.5 GB, cached after the first run

tokenizer = Qwen3Tokenizer(
    tokenizer_file_path=Path("qwen3") / "tokenizer-base.json"
)
model = Qwen3Model(QWEN_CONFIG_06_B)
model.load_state_dict(torch.load(Path("qwen3") / "qwen3-0.6B-base.pth"))
model.to(device)
print("parameters:", sum(p.numel() for p in model.parameters()) / 1e9, "B")

## 3 · Predict, then check

Before running the next cell, write down your guess: how many tokens will the
model produce per second on *this* machine?

Reference: the course's training logs run at roughly **47 tokens/second** on the
author's GPU.

## 4 · Greedy generation, streaming

In [ ]:
from reasoning_from_scratch.ch02 import generate_text_basic_stream_cache

prompt = "Explain what a reasoning model is in two sentences."
ids = torch.tensor(tokenizer.encode(prompt), device=device).unsqueeze(0)

text = []
for tok in generate_text_basic_stream_cache(
    model, ids, max_new_tokens=120, eos_token_id=tokenizer.eos_token_id
):
    text.append(tok.item())
    print(tokenizer.decode([tok.item()]), end="", flush=True)

print()
print("\ngenerated tokens:", len(text))

**What you should see:** fluent, unremarkable prose. This is the **base** model
before any reasoning training — it does not think before answering. That is the
baseline you improve in chapters 4-8.

## 5 · Benchmark: with and without the KV cache

In [ ]:
import time
from reasoning_from_scratch.ch02 import (
    generate_text_basic_stream, generate_text_basic_stream_cache,
)

prompt = "List the first ten prime numbers and explain the sieve of Eratosthenes."
ids = torch.tensor(tokenizer.encode(prompt), device=device).unsqueeze(0)

def bench(fn, label, n=200):
    t0 = time.perf_counter()
    ntok = 0
    for _ in fn(model, ids, max_new_tokens=n, eos_token_id=tokenizer.eos_token_id):
        ntok += 1
    dt = time.perf_counter() - t0
    print(f"{label:22} {ntok:4d} tokens  {dt:6.2f}s  {ntok/dt:6.1f} tok/s")
    return ntok / dt

torch.manual_seed(0)
slow = bench(generate_text_basic_stream, "no cache", n=60)   # TRY: n=120
torch.manual_seed(0)
fast = bench(generate_text_basic_stream_cache, "KV cache")

print(f"\nspeed-up: {fast/slow:.1f}x")

**TRUTH CHECK.** If your speed-up is not several times, something is off: check
that `max_new_tokens` is the same in both calls, and that the model is on the
device you think it is.

**Why it is not infinite:** the cached version still runs one forward pass per
token. What it skips is recomputing keys and values for the prefix.

## 6 · Compute the cache size instead of guessing

The model config gives you everything needed.

In [ ]:
cfg = QWEN_CONFIG_06_B
per_token = cfg["n_kv_groups"] * cfg["head_dim"] * 2 * cfg["n_layers"] * 2  # 2 = K and V, 2 = bytes/bf16

print(f"per token: {per_token} bytes = {per_token/1024:.0f} KiB")
for n in (512, 1024, 2048, 8192, cfg["context_length"]):
    print(f"{n:>6} tokens -> {n*per_token/1024/1024:>8.0f} MiB")

no_gqa = cfg["n_heads"] * cfg["head_dim"] * 2 * cfg["n_layers"] * 2
print(f"\nwith {cfg['n_heads']} KV heads instead of {cfg['n_kv_groups']}: "
      f"{no_gqa/per_token:.0f}x larger")

**Takeaway to remember:** 112 KiB per token. At the full 40,960-token context
that is ~4.4 GiB for a *single* sequence — for a 0.6B model. This is why
grouped-query attention exists, and why chapter 6's 1,024-token reasoning traces
fill a GPU.

## 7 · Exercises

1. Set `n_layers=27` in a hand-built `KVCache` and run generation. What breaks,
   and what does the error tell you?
2. Time time-to-first-token for a 10-token prompt vs a 500-token prompt. Explain
   the shape.
3. Try `torch.compile`: load with `load_model_and_tokenizer(..., use_compile=True)`
   and compare tokens/sec after the first call.